# 23 — Production Reliability, SRE, Release Gates & Capstone

Đây là graduation notebook cuối cùng. Security foundations 16–22 là prerequisite.

## Learning requirements
- async/concurrency/timeouts/cancellation;
- retry/backoff/fallback/circuit breaker;
- idempotency and duplicate-side-effect prevention;
- rate limits, budgets and backpressure;
- SLI/SLO and operational dashboards;
- deployment/canary/rollback;
- functional + security evaluation as release gates;
- backup/recovery for durable state and memory;
- incident runbook integration.

Production-ready != deployed. Production-ready nghĩa là hệ thống có thể fail, recover, rollback, observe và contain risk có kiểm soát.

## Failure taxonomy

```text
Request
  |- LLM provider failure
  |- Tool/API failure
  |- MCP failure
  |- DB/checkpointer/store failure
  |- retrieval degradation
  |- malformed structured output
  |- policy/security block
  |- human approval timeout
  `- logical agent failure / loop
```

Với mỗi class define timeout, retryability, fallback, user-visible status, alert và recovery behavior.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class FailureClass(str, Enum):
    TRANSIENT = "transient"
    PERMANENT = "permanent"
    SECURITY = "security"

@dataclass(frozen=True)
class RetryDecision:
    retry: bool
    max_attempts: int
    fallback: str | None

def retry_policy(kind: FailureClass) -> RetryDecision:
    if kind == FailureClass.TRANSIENT:
        return RetryDecision(True, 3, "fallback_provider")
    if kind == FailureClass.SECURITY:
        return RetryDecision(False, 0, None)
    return RetryDecision(False, 0, None)

assert retry_policy(FailureClass.TRANSIENT).retry
assert not retry_policy(FailureClass.SECURITY).retry

## Async, concurrency & backpressure

Học và benchmark:
- `ainvoke`, `astream`;
- async tools;
- parallel independent tasks;
- bounded semaphore/concurrency;
- per-provider and per-tool quotas;
- cancellation propagation;
- queue/backpressure;
- request and per-run budgets.

Không parallelize dependent stages. Đo provider quota, duplicated context và p95 latency trước/sau.

## SLI / SLO

Suggested SLIs:
- task success rate;
- p50/p95 end-to-end latency;
- TTFT;
- tool failure rate;
- model fallback rate;
- resume success rate;
- cost per successful task;
- approval wait time;
- security policy violation rate;
- attack success rate from online/offline security evals.

Define SLO based on product requirement, không copy một con số generic.

## Release gate

```text
Code / Prompt / Model / Tool / Skill / MCP change
                    |
                    v
               Unit tests
                    v
             Integration tests
                    v
             Functional agent eval
                    v
              Security eval
                    v
              Load/resilience test
                    v
                Canary deploy
                    v
             Production monitoring
```

Model ID, prompt, tool schema/description, skill và MCP changes đều phải được xem là behavior-affecting changes.

# Capstone — Production AI Interview Agent / Codebase Interview System

## Target flow
```text
Wizard: project name + goal
          -> Source Scanner
          -> Project Understanding
          -> Interview Supervisor
                |- Scanner specialist
                |- Requirement analyst
                `- Question specialist
          -> Interview State
          -> Ask free-text / choices
          -> validate + HITL where required
          -> Final Structured Specification
```

## Required system capabilities
### Agent engineering
- LangChain models/messages/tools/structured output/middleware/streaming;
- LangGraph state, conditional edges, persistence, interrupts, resume, subgraphs;
- RAG with retrieval evaluation and evidence;
- memory scopes and tenant isolation;
- multi-agent/Deep Agents only where benchmark justifies them;
- MCP tools with approved-server/tool policies;
- LangSmith tracing + offline/online evaluation.

### Security
- threat model with OWASP Agentic coverage;
- deterministic authorization and guardrails;
- prompt-injection and memory-poison defenses;
- sandbox/MCP/supply-chain controls;
- adversarial security regression dataset;
- audit, kill switch and incident-response runbook.

### Reliability
- retries/fallbacks only for retryable failures;
- idempotent side effects;
- bounded async/concurrency;
- budgets/rate limits;
- SLOs/alerts;
- canary + rollback;
- backup/restore for durable state where required.

## Required artifacts

```text
artifacts/
|- context-engineering.md
|- rag-benchmark.md
|- evaluation-results.md
|- optimization-report.md
|- security/
|   |- threat-model.md
|   |- risk-register.csv
|   |- policy-matrix.md
|   |- identity-model.md
|   |- injection-defense.md
|   |- memory-write-policy.md
|   |- sandbox-design.md
|   |- mcp-trust-policy.md
|   |- supply-chain-inventory.md
|   |- security-eval-dataset.jsonl
|   |- security-evaluation-results.md
|   `- incident-response-runbook.md
`- capstone/
    |- architecture.md
    |- state-schema.md
    |- tool-catalog.md
    |- eval-dataset.jsonl
    |- runbook.md
    `- README.md
```

## Final acceptance criteria

Không graduate nếu chỉ happy-path pass. Tối thiểu phải chứng minh:
- crash + resume;
- provider/tool/MCP timeout handling;
- invalid structured output handling;
- model/tool budget enforcement;
- repeated-question prevention;
- user/tenant isolation;
- indirect prompt injection containment;
- unauthorized tool execution blocked;
- memory poisoning blocked/quarantined;
- rejected approval causes no side effect;
- duplicate side effect prevented after retry;
- kill switch and read-only emergency mode;
- functional eval threshold passed;
- security eval threshold passed;
- canary rollback procedure documented.

## Graduation definition
Bạn có thể defend architecture, reliability và security decisions bằng **tests + metrics + traces + runbooks**, không chỉ bằng diagram hoặc prompt.